# Plotting broadband SEDs

An object's broadband spectral energy distribution - one brightness measurement per band,
plotted against each band's wavelength - is a quick way to eyeball an object's color.

We read the values straight off the object's own per-band columns, like `u_psfMag` and `g_psfMag`.

If you'd like to do anything more complicated, like many objects on a single grid, you should
consider your own plotting methods.

By default, our methods use the [Rubin-recommended palettes](https://rtn-045.lsst.io/#colorblind-friendly-plots),
which try to make the plots colorblind-friendly.

## Fetch some catalog data

We will use mock EDP2 object data. This data is entirely generated - it follows the schema of the
`object_lc` catalog of the DP2 release candidate, but no real Rubin measurement is in here.

Because the values are random, the SEDs below are noise rather than science - expect
generous error bars and no meaningful color.

The columns we care about are the per-band ones: `<band>_psfMag` holds the magnitude, and
`<band>_psfMagErr` the uncertainty that goes with it.

In [ ]:
import lsdb
from lsdb_rubin.plot_sed import plot_sed

In [ ]:
objects = lsdb.open_catalog("../../tests/data/mock_dp2_object_20")
objects = objects.compute()
objects[["coord_ra", "coord_dec", "refBand"] + [f"{band}_psfMag" for band in "ugrizy"]]

## Basic usage

You MUST provide a single object as the first argument. This is the only required argument, however, and we will use the `psfMag` columns by default.

Note the trailing `;` on the cells below. `plot_sed` returns the axes it drew on, and without the semicolon Jupyter prints the `<Axes: ...>` repr underneath the plot.

In [ ]:
plot_sed(objects.iloc[3]);

## Using a different brightness

Any per-band measurement works, as long as the catalog names it `<band>_<something>`. Pass that
`<something>` and we will find the columns. The uncertainty is read from `<band>_<something>Err`
when the catalog has one - a measurement without an error column is drawn as a bare point.

You can use either a magnitude or a flux column. The y-axis will be automatically inverted when plotting magnitudes.

In [ ]:
plot_sed(objects.iloc[3], mag_field="cModelMag");

In [ ]:
plot_sed(objects.iloc[3], flux_field="psfFlux");

## Incomplete band data

If the object has no measurement in one or more bands, those will not appear in the plot or the
legend. The following object was only measured in the `r` and `y` bands.

In [ ]:
missing_bands = objects.query("objectId == 8382607273521735")
plot_sed(missing_bands.iloc[0]);

## Customizing appearance

While we use the default Rubin color-blind friendly palatte, that might not work for you for
whatever reason. For visual aspects of the plot that will vary by band, you can pass in a
dictionary that maps the band string to the desired value.

- `filter_colors`
- `filter_symbols`

Note that there is no `filter_linestyles` here, unlike when plotting a light curve. Each band is a
single point, so there is no segment between points for a line style to draw.

The following example uses an alternative built-in set of plot symbols and a custom color palette.

In [ ]:
from lsdb_rubin.bands import plot_filter_symbols

color_map = {
    "y": "#0c71ff",  # Blue
    "z": "#49be61",  # Green
    "i": "#ff0000",  # Red
    "r": "#ffc200",  # Orange/Yellow
    "g": "#f341a2",  # Pink/Magenta
    "u": "#990099",  # Purple
}

plot_sed(
    objects.iloc[3],
    filter_colors=color_map,
    filter_symbols=plot_filter_symbols,
);

You can also override a single value of the existing by-filter maps. In this case, we're marking
only the `r` band with a different symbol.

In [ ]:
from lsdb_rubin.bands import plot_filter_symbols

plot_sed(objects.iloc[3], filter_symbols=plot_filter_symbols | {"r": "X"});

## Bands and wavelengths

`band_names` picks which bands to look for, and in what order - handy to restrict the plot to a
subset. These are the *column prefixes*, so `["g", "r", "i"]` reads `g_psfMag`, `r_psfMag`, and
`i_psfMag`.

`band_wavelengths` sets where each band lands on the x-axis, and `band_widths` how wide its bar
is - both in nanometers, always, whatever `x_units` you plot in. The defaults are extracted from
the [official Rubin LSSTCam information](https://lsstcam.lsst.io): the
wavelength of each LSST passband, and the full width at half maximum of its throughput.

In [ ]:
plot_sed(objects.iloc[3], band_names=["g", "r", "i"]);

## X-axis units

By default the x-axis is the wavelength of each band, in nanometers. `x_units`
switches that to any other unit on the spectral axis, using
[astropy's spectral equivalencies](https://docs.astropy.org/en/stable/units/equivalencies.html):
another wavelength (`"angstrom"`, `"micron"`), a frequency (`"THz"`, `"GHz"`), an energy
(`"eV"`, `"keV"`), or a wavenumber (`"1/cm"`). An ``astropy`` unit object works as well as its name.

Both axes are left linear, and `plot_sed` draws on the axes you pass as `ax` - or on the current
ones - so you can scale or invert either axis with matplotlib after the call.

In [ ]:
plot_sed(objects.iloc[3], x_units="angstrom");

In [ ]:
plot_sed(objects.iloc[3], x_units="THz");

## Y-axis units

By default the y-axis holds the values as the catalog stores them - nanojanskys for a flux column,
AB magnitudes for a magnitude column. `y_units` converts them, at each band's own wavelength, using
[astropy's spectral flux density equivalencies](https://docs.astropy.org/en/stable/units/equivalencies.html#spectral-flux-and-luminosity-density-units).

Pass `"nJy"`, `"ABmag"`, a shorthand like `"FLAM"`, or any unit astropy can reach, such as
`u.erg / u.s / u.cm**2 / u.AA`. The axis is inverted whenever what it carries is a magnitude, so
plotting a flux column in `"ABmag"` flips it, and plotting a magnitude column in `"nJy"` does not.

In [ ]:
plot_sed(objects.iloc[3], mag_field="psfMag");

In [ ]:
import astropy.units as u

plot_sed(objects.iloc[3], flux_field="psfFlux", y_units=u.erg / u.s / u.cm**2 / u.AA);

## Composing with other plots

`plot_sed` takes the axes to draw on as `ax`, and hands them back, so it slots into a subplot
grid next to any other plot - here, alongside the object's forced-source light curve.

In [ ]:
import matplotlib.pyplot as plt

from lsdb_rubin.plot_light_curve import plot_light_curve

figure, axes = plt.subplots(1, 2, figsize=(13, 4))

plt.sca(axes[0])
plot_light_curve(objects.iloc[3]["objectForcedSource"])

plot_sed(objects.iloc[3], ax=axes[1])

figure.tight_layout();

## About

**Authors**: Sandro Campos

Last updated on: Aug 27, 2026

If you use `lsdb` for published research, please cite following the [instructions here](https://docs.lsdb.io/en/stable/citation.html).